# 01. 特徴量パイプライン (M5 Accuracy)

全モデル共通の特徴量を生成し、Driveに保存する。
- 入力: `MyDrive/Colab Data/m5-forecasting-accuracy/input/*.csv`
- 出力: `MyDrive/Colab Data/m5-forecasting-accuracy/cache/`
  - `features_all.parquet`  全特徴量付きDataFrame
  - `val_folds.pkl`         5-fold CV定義
  - `feature_cols.pkl`      特徴量列リスト

In [ ]:
import gc, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
print(f'pandas: {pd.__version__}')
T0 = time.time()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE = Path('/content/drive/MyDrive/Colab Data/m5-forecasting-accuracy')
INPUT = DRIVE / 'input'
CACHE = DRIVE / 'cache'
CACHE.mkdir(parents=True, exist_ok=True)
for f in ['calendar.csv','sell_prices.csv','sales_train_evaluation.csv','sample_submission.csv']:
    p = INPUT / f
    print(f'{f}: {"OK" if p.exists() else "MISSING"}')

## 1. 生データ読込 + melt

In [ ]:
calendar = pd.read_csv(INPUT/'calendar.csv', parse_dates=['date'])
calendar['d_num'] = calendar['d'].str[2:].astype(int)

id_cols = ['id','item_id','dept_id','cat_id','store_id','state_id']
d_cols  = [f'd_{i}' for i in range(1,1942)]
dt = {c:'int16' for c in d_cols}; dt.update({c:'category' for c in id_cols})
sales = pd.read_csv(INPUT/'sales_train_evaluation.csv', dtype=dt)
# eval期間プレースホルダ
for d in range(1942,1970): sales[f'd_{d}'] = np.int16(0)
print('sales:', sales.shape)

In [ ]:
all_d = [f'd_{i}' for i in range(1,1970)]
df = sales.melt(id_vars=id_cols, value_vars=all_d, var_name='d', value_name='sales')
df['d_num'] = df['d'].str[2:].astype('int16')
df.drop(columns=['d'], inplace=True)
df['sales'] = df['sales'].astype('int16')
del sales; gc.collect()
print('long:', df.shape, f'mem={df.memory_usage(deep=True).sum()/1024**2:.0f}MB')

In [ ]:
cal_keep = ['d_num','date','wm_yr_wk','wday','month','year','event_name_1','event_type_1','snap_CA','snap_TX','snap_WI']
df = df.merge(calendar[cal_keep], on='d_num', how='left')
for c in ['wday','month','snap_CA','snap_TX','snap_WI']: df[c] = df[c].astype('int8')
df['year'] = df['year'].astype('int16')
for c in ['event_name_1','event_type_1']: df[c] = df[c].astype('category')
print('merged:', df.shape, f'mem={df.memory_usage(deep=True).sum()/1024**2:.0f}MB')

## 2. 直近データへ絞る (d_1300以降, 約 21M 行)

フルパッケージなので学習データを `d_1300` まで拡大（前回より +200日）

In [ ]:
df = df[df['d_num'] >= 1300].reset_index(drop=True)
gc.collect()
print('filtered:', df.shape, f'mem={df.memory_usage(deep=True).sum()/1024**2:.0f}MB')

## 3. カレンダー特徴量

In [ ]:
df['day_of_month'] = df['date'].dt.day.astype('int8')
df['week_of_year'] = df['date'].dt.isocalendar().week.astype('int8')
df['quarter'] = df['date'].dt.quarter.astype('int8')
df['is_weekend'] = df['wday'].isin([1,2]).astype('int8')
df['has_event'] = df['event_name_1'].notna().astype('int8')
# event名/typeをcode化（ツリー系で使用可能）
df['event_name_code'] = df['event_name_1'].cat.codes.astype('int16')
df['event_type_code'] = df['event_type_1'].cat.codes.astype('int16')
# SNAP集約
df['snap'] = 0
for s in ['CA','TX','WI']:
    m = df['state_id']==s; df.loc[m,'snap'] = df.loc[m,f'snap_{s}']
df['snap'] = df['snap'].astype('int8')
df.drop(columns=['snap_CA','snap_TX','snap_WI','event_name_1','event_type_1','date'], inplace=True)
gc.collect()
print(f'after calendar mem={df.memory_usage(deep=True).sum()/1024**2:.0f}MB')

## 4. 価格特徴量

In [ ]:
sp = pd.read_csv(INPUT/'sell_prices.csv')
df = df.merge(sp[['store_id','item_id','wm_yr_wk','sell_price']], on=['store_id','item_id','wm_yr_wk'], how='left')
df['sell_price'] = df['sell_price'].astype('float32')
# 派生
stat = sp.groupby(['store_id','item_id'])['sell_price'].agg(price_max='max', price_min='min', price_mean='mean', price_std='std').reset_index()
df = df.merge(stat, on=['store_id','item_id'], how='left')
df['price_discount'] = ((df['price_max']-df['sell_price'])/df['price_max']).astype('float32')
df['price_norm']     = ((df['sell_price']-df['price_mean'])/df['price_mean']).astype('float32')
df['price_z']        = ((df['sell_price']-df['price_mean'])/df['price_std']).astype('float32')
for c in ['price_max','price_min','price_mean','price_std']: df[c] = df[c].astype('float32')
del sp, stat; gc.collect()
print(f'after price mem={df.memory_usage(deep=True).sum()/1024**2:.0f}MB')

## 5. ラグ・ローリング特徴量

予測期間28日に対応するため `lag >= 28`、ローリングは `shift(28)` 後に窓計算（リーク回避）

In [ ]:
df = df.sort_values(['id','d_num']).reset_index(drop=True)
# ラグ
for lag in [28,29,30,35,42,49,56,63,70]:
    df[f'lag_{lag}'] = df.groupby('id', observed=True)['sales'].shift(lag).astype('float32')
    print(f'  lag_{lag} done')
gc.collect()

In [ ]:
# ローリング (shift 28 後に rolling)
sh = df.groupby('id', observed=True)['sales'].shift(28)
for w in [7,14,28,60,180]:
    df[f'rmean_{w}'] = sh.groupby(df['id'], observed=True).transform(lambda x: x.rolling(w, min_periods=1).mean()).astype('float32')
    print(f'  rmean_{w} done')
for w in [7,14,28]:
    df[f'rstd_{w}'] = sh.groupby(df['id'], observed=True).transform(lambda x: x.rolling(w, min_periods=1).std()).astype('float32')
    print(f'  rstd_{w} done')
# 中央値・最大
df['rmedian_28'] = sh.groupby(df['id'], observed=True).transform(lambda x: x.rolling(28, min_periods=1).median()).astype('float32')
df['rmax_28']    = sh.groupby(df['id'], observed=True).transform(lambda x: x.rolling(28, min_periods=1).max()).astype('float32')
del sh; gc.collect()

## 6. グループ集約特徴量 (店舗 / アイテム / dept)

リーク回避のため 学習データのみで集約 → df 全体に merge

In [ ]:
# 学習期間（d <= 1913）のみで集約
trn = df[df['d_num']<=1913]
# store × dow
g1 = trn.groupby(['store_id','wday'], observed=True)['sales'].mean().rename('store_dow_mean').reset_index()
# item × dow
g2 = trn.groupby(['item_id','wday'], observed=True)['sales'].mean().rename('item_dow_mean').reset_index()
# dept × dow
g3 = trn.groupby(['dept_id','wday'], observed=True)['sales'].mean().rename('dept_dow_mean').reset_index()
# store × item 平均
g4 = trn.groupby(['store_id','item_id'], observed=True)['sales'].agg(si_mean='mean', si_std='std').reset_index()
del trn; gc.collect()
df = df.merge(g1, on=['store_id','wday'], how='left')
df = df.merge(g2, on=['item_id','wday'], how='left')
df = df.merge(g3, on=['dept_id','wday'], how='left')
df = df.merge(g4, on=['store_id','item_id'], how='left')
for c in ['store_dow_mean','item_dow_mean','dept_dow_mean','si_mean','si_std']:
    df[c] = df[c].astype('float32')
del g1,g2,g3,g4; gc.collect()
print(f'after groups mem={df.memory_usage(deep=True).sum()/1024**2:.0f}MB')

## 7. カテゴリエンコーディング

In [ ]:
for col in ['store_id','item_id','dept_id','cat_id','state_id']:
    df[f'{col}_enc'] = df[col].astype(str).factorize()[0].astype('int16')
print('final:', df.shape, f'mem={df.memory_usage(deep=True).sum()/1024**2:.0f}MB')

## 8. 5-fold CV 定義 + 特徴量列リスト

In [ ]:
val_folds = [
    {'fold':1, 'val_d_start':1886, 'val_d_end':1913},  # 直近28日
    {'fold':2, 'val_d_start':1858, 'val_d_end':1885},
    {'fold':3, 'val_d_start':1830, 'val_d_end':1857},
    {'fold':4, 'val_d_start':1802, 'val_d_end':1829},
    {'fold':5, 'val_d_start':1774, 'val_d_end':1801},
]
feature_cols = [
    'wday','month','year','day_of_month','week_of_year','quarter','is_weekend','has_event','event_name_code','event_type_code','snap',
    'sell_price','price_max','price_min','price_mean','price_std','price_discount','price_norm','price_z',
    'lag_28','lag_29','lag_30','lag_35','lag_42','lag_49','lag_56','lag_63','lag_70',
    'rmean_7','rmean_14','rmean_28','rmean_60','rmean_180','rstd_7','rstd_14','rstd_28','rmedian_28','rmax_28',
    'store_dow_mean','item_dow_mean','dept_dow_mean','si_mean','si_std',
    'store_id_enc','item_id_enc','dept_id_enc','cat_id_enc','state_id_enc',
]
print(f'features: {len(feature_cols)}')

## 9. Driveに保存

In [ ]:
# 学習に必要な列のみ残してparquet保存（メモリ節約）
keep = ['id','d_num','sales','store_id','item_id','dept_id','cat_id','state_id'] + feature_cols
df_save = df[keep].copy()
del df; gc.collect()
print('saving features.parquet...')
df_save.to_parquet(CACHE/'features_all.parquet', index=False, compression='snappy')
print(f'  size={ (CACHE/"features_all.parquet").stat().st_size/1024**2:.0f}MB')

with open(CACHE/'val_folds.pkl','wb') as f: pickle.dump(val_folds, f)
with open(CACHE/'feature_cols.pkl','wb') as f: pickle.dump(feature_cols, f)
print(f'\n全工程: {(time.time()-T0)/60:.1f}分')
print('=== 完了 ===')